# GN on jet engine bracket GINN

### Imports and setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.wire import ConditionalWIRE 
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.visualization.utils_mesh import get_mesh
from GINN.ph.ph_manager import PHManager
from models.net_w_partials import NetWithPartials
from training.modular.residuals import ResidualLibrary, ResidualTerm, compute_loss
from training.modular.optimizers import GaussNewton

torch.manual_seed(0)

device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

### 1) Load model

In [ ]:
# model = ConditionalWIRE([4, 64, 64, 64, 1], False, 18, 1, 6)
# run_id = "2025_04_25__11_43_40-qc0rhngg"

model = ConditionalWIRE([4, 32, 32, 32, 1], False, 18, 1, 6)
run_id = "2025_07_21__13_25_49-rrzz9q06"
model.load_state_dict(torch.load(f"GINN_models/{run_id}/{run_id}-model.pt", map_location=device)["state_dict"])

In [ ]:
## Plot
bounds = torch.from_numpy(np.load('Data JEB/bounds.npy')).float()

f = lambda x : model(x, 0.05+torch.zeros([len(x), 1]))
verts, faces = get_mesh(
    f, 
    N=128, 
    device=device,
    bbox_min=bounds[:,0],
    bbox_max=bounds[:,1],
    chunks=1
)
verts_init, faces_init = verts, faces

# fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
# fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
# fig.display()

### 2) Prepare point samples

In [ ]:
pts_interface_all = torch.from_numpy(np.load('Data JEB/interface_points.npy')).to(device, dtype=torch.float64)
pts_nearout_all = torch.from_numpy(np.load('Data JEB/pts_outside.npy')).to(device, dtype=torch.float64)
pts_farout_all = torch.from_numpy(np.load('Data JEB/pts_far_outside.npy')).to(device, dtype=torch.float64)
pts_out_all = torch.vstack([pts_nearout_all, pts_farout_all]).to(device, dtype=torch.float64)
pts_inside_all = torch.from_numpy(np.load('Data JEB/pts_inside.npy')).to(device, dtype=torch.float64)
# print(pts_outside_all.shape)

In [ ]:
# fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
# fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
# fig += k3d.points(pts_out_all[::10], color=0xff0000, point_size=0.01)
# # fig += k3d.points(pts_inside_all[::10], color=0x00ff00, point_size=0.01)
# # fig += k3d.points(pts_farout_all[::10], color=0x0000ff, point_size=0.01)
# # fig += k3d.points(pts_outside_all)
# fig.display()

In [ ]:
def subsample(pts, count):
    idx = torch.randperm(pts.size(0))[:count]
    return pts[idx]

def distance_filter(pts, pts_filter, th_dist=0.025, mode="leave_far"):
    dist = torch.min(torch.norm(pts[:, None, :] - pts_filter[None, :, :], dim=2), dim=1)[0]
    if mode=="leave_near":
        return pts[dist < th_dist]
    return pts[dist > th_dist]

def inside_mask(pts, bounds):
    return ((pts >= bounds[:, 0]) & (pts <= bounds[:, 1])).all(dim=1)

def inside_filter(pts, bounds):
    return pts[inside_mask(pts, bounds)]

In [ ]:
model = model.double()
bounds = bounds.to(device, dtype=torch.float64)
pts_surface = sample_model_surface_binsearch(model, torch.zeros([0,3]), bounds=bounds*1.3, z=0.05+torch.zeros([1,1], dtype=torch.float64))
pts_surface_inside = inside_filter(pts_surface, bounds)
pts_surface_filtered = distance_filter(pts_surface_inside, subsample(pts_interface_all, 10_000))


fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
# fig += k3d.points(pts_surface.cpu(), color=0xff0000, point_size=0.01)
# fig += k3d.points(pts_surface_inside.cpu(), color=0x0000ff, point_size=0.01)
# fig += k3d.points(pts_surface_filtered.cpu(), color=0x00ff00, point_size=0.01)
# fig += k3d.points(subsample(pts_interface_all, 10_000).cpu(), color=0x0000ff, point_size=0.01)
# fig += k3d.points(subsample(pts_nearout_all, 1000).cpu(), color=0x00bb00, point_size=0.01)
fig.display()

### 4) Connectedness

In [ ]:
netp = NetWithPartials.create_from_model(model.float(), nz=1, nx=3)
## NOTE: the PHManager is implemented in a way where it loops over shapes, so it somehow expects a list of latent codes.

## Define the function to check if query points a are inside the envelope, which in this case is trivially the bounds.
## Since the query points will be in the bounds per construction, we could've also just returned True without checking.
def is_inside_envelope(pts: torch.Tensor):
    """Get mask for points which are inside the envelope"""
    return inside_mask(pts, bounds)

ph_manager = PHManager(
    mp_pool = None, ## we wont need parallelization
    nx = 3, ## dimension of input
    bounds = bounds, ## bounds of the rectangular domain, list of (min, max) pairs for each coordinate
    netp = netp, ## model in the functional form
    scc_n_grid_points = 128, ## nof grid points along each dimension. this grid is used to perform the discrete PH calculation
    func_inside_envelope = is_inside_envelope, ## a function to mask the points outside the envelope as these would create illegal connections. In the obstacle problem, we do not really need this.
    ph_1_hole_level = 0.1, ## persistence of holes. Ignore this if ph_loss_maxdim=0 (only 0-PH ie connectedness is optimized for)
    ph_loss_target_betti = [1,0,0], ## Target Betti-numers for each dim-PH. Connectedness means 1 for dim=0. Others can be ignored if ph_loss_maxdim=0.
    iso_level = 0, ## level defining the boundary. This also enforces the thickness of the thinest feature for an SDF. NOTE 
    ph_loss_sub0_points = False,
    ph_loss_super0_points = False,
    simjeb_root_dir = "", ## leave empty
    problem_str= "", ## leave empty
    ph_loss_maxdim=0, ## connectedness only
    is_density=False, ## is the implicit function a density field? (no for an SDF)
)

# ## NOTE: see the comments in ph_manager.calc_ph_loss_cripser on how to return what you need.
success, loss_scc, _, _ = ph_manager.calc_ph_loss_cripser(z=0.05+torch.zeros([1,1])) ## NOTE: the comment at the start of the cell.
print(success, loss_scc)

In [ ]:
success, x_in = ph_manager.calc_ph_loss_cripser(z=0.05+torch.zeros([1,1]), return_for_GN=True)
if success and len(x_in):
    loss = torch.clamp(0 - f(x_in), max=0.).pow(2).sum()
    print(loss)

### 5) Residuals

In [ ]:
res_lib = ResidualLibrary(model)

## Select the residual terms defining the specific problem
## (incl. weight, points, and (optional) target values)
## Points will be overwritten in each loop. Only specify them here to run the debug code below
pts_eikonal = torch.vstack([subsample(pts_inside_all, 1000), subsample(pts_out_all, 1000)])
res_terms = {
    "interface":    ResidualTerm(res_lib._data,             10,     subsample(pts_interface_all, 1000), vals=0),
    "design_region":ResidualTerm(res_lib._design_region,    1.0,    subsample(pts_nearout_all, 1000)),
    "eikonal":      ResidualTerm(res_lib._eikonal,          0.01,    pts_eikonal),
    "strain":       ResidualTerm(res_lib._strain,           0.0005,  subsample(pts_interface_all, 1)),
    "connectedness":ResidualTerm(res_lib._connectedness,    10,    x_in.to(device=device, dtype=torch.float64)),
    # "if_normal":    ResidualTerm(res_lib._normal,           1,      subsample(pts_interface_all, 1000), vals=0), ## ignore normals because they don't have a huge effect
}

## rrzz
#             scale     lambda      weight
# interface   -         1           1
# design      0.1       1           0.1
# eikonal     0.1       1           0.1
# strain      0.001     1           0.001
# connected   -         100         100
# normal      1         1           1


## from ALM ablation (more finely tuned) https://wandb.ai/abra/topginn/runs/im7rt3b6/
#             scale     lambda      weight
# interface   -         10          10
# design      -         1           1
# eikonal     0.01      1           0.01
# strain      0.0005    1           0.0005
# connected   -         10          10
# normal      1         1           1


## Example
model = model.double()
params = dict(model.named_parameters())

loss, unweighted_losses = compute_loss(params, res_terms, return_unweighted_losses=True)
print("weighted sum:", loss.item())
for key, l in unweighted_losses.items():
    print(key, l.item())

In [ ]:
## from the rrzz run
#               w. loss       unw. loss       \lambda_i       \mu_i       
# interface     0.5           0.01            50              0.1         
# design_region 0.2           0.002           40              0.2         
# eikonal       3.5           0.07            50              0.04        
# strain        0.4           0.4             1.00           (1.00)       
# connectedness 0.01          0.0002          120             0.4         

### 6) Train

In [ ]:
model = model.double()
params = dict(model.named_parameters())
losses_over_time = {k: [] for k in res_terms.keys()}
best_loss = float('inf')

optim = GaussNewton(model, res_terms, lr=2e-2, regularization=1e-6, do_line_search=False)
start_time = time.time()
current_time = 0

for i in (pbar:=trange(100)):
    # if current_time > 1200:
    #     break
    optim.zero_grad()


    ## Sample new points
    ## Subsample
    res_terms["interface"].points = subsample(pts_inside_all, 4096)
    res_terms["eikonal"].points = torch.vstack([subsample(pts_inside_all, 2048//2), subsample(pts_out_all, 2048//2)])
    ## Design region
    ## since the design region loss is inactive for negative points, we can filter these to not waste GN
    res_terms["design_region"].points = subsample(pts_nearout_all, 4*16384) 
    mask = res_terms["design_region"].eval(params).squeeze().nonzero().squeeze(-1)
    res_terms["design_region"].points = res_terms["design_region"].points[mask]
    if len(res_terms["design_region"].points)==0:
        res_terms["design_region"].points = torch.zeros([1,3], dtype=torch.float64)
    ## Surface
    pts_surface = sample_model_surface_binsearch(model, torch.zeros([0,3]), bounds=bounds, z=0.05+torch.zeros([1,1], dtype=torch.float64))
    pts_surface = inside_filter(pts_surface, bounds)
    pts_surface = distance_filter(pts_surface, subsample(pts_interface_all, 8192), 0.04)
    res_terms["strain"].points = pts_surface
    
    ## Evaluate the losses
    loss, unweighted_losses = compute_loss(params, res_terms, return_unweighted_losses=True)
        
    loss.backward()
    
    with torch.no_grad():
    #     loss_metric = unweighted_losses["data"] + unweighted_losses["mean_curvature"]

    #     current_time = time.time() - start_time
    #     loss_over_time[current_time] = loss_metric.item()
    #     # distance_over_time[current_time] = compute_distance(model.double(), catenoid_level_set, pts_surface, pts_surface_true, 1.0)
    #     # chamfer_over_time[current_time] = chamfer_div(model, pts_surface_true)

    #     if loss_metric.item() < best_loss:
    #         best_loss = loss_metric.item()
    #         best_model_state = copy.deepcopy(model.state_dict())
    
        for k, v in unweighted_losses.items():
            losses_over_time[k].append(v.item())

        unweighted_losses_str = " ".join(f"{key}: {l.item():.2e}" for key, l in unweighted_losses.items())
        pbar.set_description(unweighted_losses_str + " "
                            f"nof_pts: {len(pts_surface)}"
                            )
    optim.step()

fig, axs = plt.subplots(1,2, figsize=(12,4))
for k, v in losses_over_time.items():
    axs[0].plot(v, label=k)
    axs[1].plot(np.array(v)*res_terms[k].weight, label=k)
axs[0].semilogy()
axs[1].semilogy()
axs[0].set_title("unweighted")
axs[1].set_title("weighted")
axs[0].legend()
plt.show()

In [129]:
## Save model
# torch.save(model.state_dict(), f"GINN_models/{run_id}/{run_id}-model-abl-100.pt")

In [ ]:
model = model.float()
f = lambda x : model(x, 0.05+torch.zeros([len(x), 1]))
verts, faces = get_mesh(
    f, 
    N=128, 
    device=device,
    bbox_min=bounds[:,0],
    bbox_max=bounds[:,1],
    chunks=1
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
## OLD
# fig += k3d.mesh(verts_init, faces_init, color=0x444444, side='double', flat_shading=False)
## NEW
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
# fig += k3d.points(pts_surface.cpu(), color=0xbb0000, point_size=0.01)
# fig += k3d.points(subsample(pts_nearout_all, 1000).cpu(), color=0x00bb00, point_size=0.01)
# fig += k3d.points(subsample(pts_farout_all, 1000).cpu(), color=0x0000bb, point_size=0.01)
# pts_aa = distance_filter(pts_surface, subsample(pts_out_all, 4*8192), mode="leave_near")
# fig += k3d.points(subsample(pts_aa, 1000).cpu(), color=0xbb00bb, point_size=0.01)
fig.display()